In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_9.pickle

trying: ['responses_df_2022', 'responses_df_2022_cell10']


me:  6
me:  6
trying: ['responses_df_2022', 'responses_df_2022_cell10']


me:  6
me:  6
trying: ['question_of_interest_cell18', 'question_name']
me:  6
me:  6
trying: ['question_of_interest_cell18', 'question_name']
me:  6
me:  6
trying: ['title_for_x_axis', 'title_for_x_axis_cell20']
me:  6
me:  8
trying: ['title_for_x_axis', 'title_for_x_axis_cell20']
me:  6
me:  8
trying: ['file_name_2018', 'file_name_2017']
me:  1
me:  1
trying: ['file_name_2018', 'file_name_2017']
me:  1
me:  1
trying: ['count_then_return_percent_21']
me:  9
trying: ['percentages_per_country_df']
me:  4
trying: ['create_dataframe_of_counts_16']
me:  4
trying: ['question_of_interest_cell21']
me:  9
trying: ['replace_hyphen_with_en_dash']
me:  2
trying: ['responses_df_2019_cell10']


me:  2
trying: ['responses_df_2017']


me:  6
trying: ['go']
me:  0
trying: ['percentages']
me:  5
trying: ['count_then_return_percent_18']
me:  6
trying: ['glob']
me:  0
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  6
trying: ['percentages_cell19']
me:  7
trying: ['base_dir_2018']
me:  1
trying: ['question_of_interest_cell19']
me:  7
trying: ['base_dir_2022']
me:  1
trying: ['count_then_return_percent_19']
me:  7
trying: ['base_dir_2019']
me:  1
trying: ['orig_output']
me:  10
trying: ['file_name_2019']
me:  1
trying: ['country_df_combined_cell18']
me:  6
trying: ['responses_df_2020']


...

me:  1
trying: ['file_name_2020']
me:  1
trying: ['responses_df_2018_cell10']


me:  2
trying: ['file_name_2022']
me:  1
trying: ['question_of_interest_cell20']
me:  8
trying: ['load_survey_data']
me:  1
trying: ['orientation_for_chart']
me:  5
trying: ['combine_subset_of_data_from_multiple_years_20']
me:  8
trying: ['base_dir_2021']
me:  1
trying: ['responses_df_2019']


me:  1
trying: ['count_then_return_percent_17']
me:  5
trying: ['base_dir_2020']
me:  1
trying: ['pd']
me:  0
trying: ['question_of_interest']
me:  5
trying: ['responses_in_order']
me:  5
trying: ['count_then_return_percent_20']
me:  8
trying: ['file_name_2021']
me:  1
trying: ['age_df_combined']
me:  8
trying: ['px']
me:  0
trying: ['directory']
me:  1
trying: ['percentages_cell17']
me:  5
trying: ['sns']
me:  0
trying: ['warnings']
me:  0
trying: ['add_year_column_to_dataframes_20']
me:  8
trying: ['responses_df_2018']


me:  1


Declaring variable go
Declaring variable glob
Declaring variable np
Declaring variable pd
Declaring variable px
Declaring variable sns
Declaring variable warnings
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable base_dir_2018
Declaring variable base_dir_2022
Declaring variable base_dir_2019
Declaring variable file_name_2019
Declaring variable responses_df_2020
Declaring variable base_dir_2017
Declaring variable directory_cell8
Declaring variable responses_df_2021
Declaring variable file_name_2020
Declaring variable file_name_2022
Declaring variable load_survey_data
Declaring variable base_dir_2021
Declaring variable responses_df_2019
Declaring variable base_dir_2020
Declaring variable file_name_2021
Declaring variable directory
Declaring variable responses_df_2018
Declaring variable replace_hyphen_with_en_dash
Declaring variable responses_df_2019_cell10
Declaring variable responses

In [4]:
%%RecordEvent
%%time
### cell 10 ###

### cell 10 (vectorized cuDF version)

# Combines percent distributions across years, fully on GPU via cudf.pandas

def combine_subset_of_data_from_multiple_years_22(question, x_axis_title, include_2017=None):
    # unified gender mapping
    gender_map_all = {
        'Male': 'Man',
        'Female': 'Woman',
        'Nonbinary': 'Prefer to self-describe',
        'Prefer not to say': 'Prefer to self-describe',
        'A different identity': 'Prefer to self-describe',
        'Non-binary, genderqueer, or gender non-conforming': 'Prefer to self-describe'
    }
    # map each year to its DataFrame
    df_map = {
        2022: responses_df_2022_cell10,
        2021: responses_df_2021,
        2020: responses_df_2020,
        2019: responses_df_2019_cell10,
        2018: responses_df_2018_cell10,
        2017: responses_df_2017
    }
    years = [2022, 2021, 2020, 2019, 2018]
    if include_2017:
        years.append(2017)

    dfs = []
    for yr in years:
        df = df_map[yr]
        # select the right column in 2017
        col = 'GenderSelect' if (question == 'What is your gender? - Selected Choice' and yr == 2017) else question
        # wrap into cuDF DataFrame via pandas API
        tmp = pd.DataFrame(df[[col]].rename(columns={col: x_axis_title}))
        tmp['year'] = str(yr)
        dfs.append(tmp)

    # concatenate using pandas API (cudf.pandas will dispatch to cudf.concat)
    combined = pd.concat(dfs, ignore_index=True)

    # apply gender normalization
    if question == 'What is your gender? - Selected Choice':
        combined[x_axis_title] = combined[x_axis_title].replace(gender_map_all)

    # compute counts and percentages in one pass
    counts = (
        combined
        .groupby(['year', x_axis_title], sort=False)
        .size()
        .reset_index(name='count')
    )
    totals = counts.groupby('year')['count'].transform('sum')
    counts['percentage'] = (counts['count'] / totals * 100).round(1)

    # select and sort
    result = counts[['percentage', 'year', x_axis_title]]
    return result.sort_values(by=['year', 'percentage'])

# call
question_of_interest_cell22 = 'What is your gender? - Selected Choice'
title_for_x_axis_cell22 = ''
age_df_combined_cell22 = combine_subset_of_data_from_multiple_years_22(
    question_of_interest_cell22,
    title_for_x_axis_cell22,
    include_2017=True
)
age_df_combined_cell22 = age_df_combined_cell22.sort_values(by=['year', 'percentage'])
age_df_combined_cell22.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 18 entries, 15 to 0
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   percentage  18 non-null     float64
 1   year        18 non-null     object
 2               18 non-null     object
dtypes: float64(1), object(2)
memory usage: 698.0+ bytes
CPU times: user 439 ms, sys: 15.7 ms, total: 455 ms
Wall time: 454 ms


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_10_try_3.pickle

migration speed (bps): 775286385.918016
---------------------------
variables to migrate:
file_name_2017 76
directory_cell8 170
combine_subset_of_data_from_multiple_years_20 144
file_name_2020 81
percentages_per_country_df 48
count_then_return_percent_20 144
file_name_2022 81
create_dataframe_of_counts_16 144
age_df_combined 48
load_survey_data 144
title_for_x_axis_cell20 49
base_dir_2021 155
add_year_column_to_dataframes_20 144
base_dir_2020 155
combine_subset_of_data_from_multiple_years_22 144
file_name_2018 76
px 72
question_of_interest_cell22 87
file_name_2021 81
orientation_for_chart 50
sns 72
directory 163
count_then_return_percent_17 144
warnings 72
pd 72
question_of_interest 90
go 72
responses_in_order 184
np 72
age_df_combined_cell22 48
orig_output 16
percentages_cell17 48
percentages 48
responses_df_2022_cell10 48
title_for_x_axis_cell22 49
responses_in_order_cell21 120
count_then_return_percent_21 144
percentages_cell21 48
question_of_interest_cell21 87
responses_df_2018_cel

In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['responses_df_2022', 'responses_df_2019', 'responses_df_2018', 'responses_df_2021', 'responses_df_2020']
Intermediate variables ['base_dir_2020', 'base_dir_2018', 'file_name_2020', 'responses_df_2017', 'base_dir_2017', 'directory', 'file_name_2017', 'file_name_2019', 'directory_cell8', 'base_dir_2019', 'file_name_2018', 'file_name_2022', 'base_dir_2022', 'base_dir_2021', 'file_name_2021']
Future variables ['responses_df_2022_cell10', 'responses_df_2017', 'percentages']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables ['responses_df_2022', 'responses_df_2022_cell10', 'responses_df_2019_cell10']
Intermediate variables ['responses_df_2018_cell10']
Future variables ['responses_df_2017', 'percentages', 'responses_df_2018_cell10', 'responses_df_2022_cell10', 'responses_df_2021'

In [7]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_10_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[10], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
responses_df_2020_opt = responses_df_2020
responses_df_2022_cell10_opt = responses_df_2022_cell10
responses_df_2021_opt = responses_df_2021
title_for_x_axis_cell22_opt = title_for_x_axis_cell22
responses_df_2022_opt = responses_df_2022
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_10.pickle
assert compare_df(responses_df_2020_opt, responses_df_2020)
assert compare_df(responses_df_2022_cell10_opt, responses_df_2022_cell10)
assert compare_df(responses_df_2021_opt, responses_df_2021)
assert title_for_x_axis_cell22_opt == title_for_x_axis_cell22
assert compare_df(responses_df_2022_opt, responses_df_2022)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
